In [ ]:
"""One DICOM header per series -> scanner identity, geometry, sequence params.

ponytail: reads one header per series (24,371 files), not all 819,640. Everything
this table is for - fold groups, laterality, physical scale - is a property of the
series, not of the slice. Slice ordering does need every header, but that belongs
to the cache build, not here.
"""
import os, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom

T0 = time.time()
def log(m): print(f"[{time.time()-T0:6.1f}s] {m}", flush=True)

def find_root():
    for c in [Path("/kaggle/input/rsna-knee-abnormality-detection"),
              Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")]:
        if (c / "train.csv").is_file():
            return c
    for d in sorted(p for p in Path("/kaggle/input").iterdir() if p.is_dir()):
        if (d / "train.csv").is_file():
            return d
    raise FileNotFoundError("competition mount not found")

ROOT = find_root()
log(f"root: {ROOT}")

TAGS = ["Manufacturer", "ManufacturerModelName", "SoftwareVersions", "StationName",
        "MagneticFieldStrength", "ImagingFrequency", "ReceiveCoilName",
        "Laterality", "PatientSex", "PatientAge", "BodyPartExamined",
        "SeriesDescription", "ProtocolName", "ScanningSequence", "SequenceVariant",
        "ScanOptions", "MRAcquisitionType", "RepetitionTime", "EchoTime",
        "InversionTime", "FlipAngle", "EchoTrainLength", "PixelBandwidth",
        "NumberOfAverages", "SliceThickness", "SpacingBetweenSlices",
        "Rows", "Columns", "TransferSyntaxUID"]

def one_series(args):
    """Read the first slice's header, plus the slice count."""
    study, series, d = args
    files = sorted(os.listdir(d))
    row = {"StudyInstanceUID": study, "SeriesInstanceUID": series,
           "n_slices": len(files)}
    try:
        ds = pydicom.dcmread(os.path.join(d, files[0]), stop_before_pixels=True)
    except Exception as e:
        row["error"] = str(e)[:80]
        return row
    for t in TAGS:
        v = getattr(ds, t, None)
        if t == "TransferSyntaxUID":
            v = getattr(getattr(ds, "file_meta", None), "TransferSyntaxUID", None)
        row[t] = str(v) if v is not None else None
    for t, n in (("PixelSpacing", 2), ("ImagePositionPatient", 3),
                 ("ImageOrientationPatient", 6)):
        v = getattr(ds, t, None)
        for i in range(n):
            row[f"{t}_{i}"] = float(v[i]) if v is not None and len(v) > i else np.nan
    return row

jobs = []
for split in ("train_series", "test_series"):
    base = ROOT / split
    if not base.is_dir():
        continue
    for study in sorted(os.listdir(base)):
        sd = base / study
        for series in sorted(os.listdir(sd)):
            jobs.append((study, series, str(sd / series)))
log(f"{len(jobs)} series to read")

with ThreadPoolExecutor(max_workers=32) as ex:
    rows = list(ex.map(one_series, jobs))
log(f"read {len(rows)} headers")

df = pd.DataFrame(rows)
df.to_csv("series_meta.csv", index=False)
log(f"wrote series_meta.csv {df.shape}")

# --- what the table says ----------------------------------------------------- #
print("\nerrors:", int(df.get("error", pd.Series(dtype=object)).notna().sum()))
print("\ntransfer syntax:"); print(df.TransferSyntaxUID.value_counts().to_string())
print("\nmanufacturer:"); print(df.Manufacturer.value_counts(dropna=False).head(12).to_string())
print("\nPatientSex present:", int(df.PatientSex.notna().sum()), "of", len(df))
print("\nLaterality present:", int(df.Laterality.replace("", np.nan).notna().sum()),
      "of", len(df))

# The fold key the folds script cannot build locally.
key = (df.Manufacturer.fillna("?").str.strip() + "|" +
       df.ManufacturerModelName.fillna("?").str.strip())
g = df.assign(key=key).groupby("StudyInstanceUID").key.agg(lambda s: s.mode().iat[0])
print(f"\nmanufacturer|model: {g.nunique()} groups over {g.size} studies")
print((100 * g.value_counts(normalize=True)).round(1).head(12).to_string())
g.rename("scanner").to_csv("study_scanner.csv")
log("wrote study_scanner.csv")

# Laterality from geometry: image-centre x in patient coords. DICOM is LPS, +x = Left.
cx = (df.ImagePositionPatient_0
      + 0.5 * df.Columns.astype(float) * df.PixelSpacing_0 * df.ImageOrientationPatient_0
      + 0.5 * df.Rows.astype(float) * df.PixelSpacing_1 * df.ImageOrientationPatient_3)
df["centre_x"] = cx
tag = df.Laterality.replace("", np.nan)
ok = tag.notna() & cx.notna()
if ok.sum():
    pred = np.where(cx[ok] > 0, "L", "R")
    print(f"\ngeometry vs tag: {(pred == tag[ok].str.upper().str[0]).mean():.3%} "
          f"on {int(ok.sum())} tagged series")
df[["StudyInstanceUID", "SeriesInstanceUID", "Laterality", "centre_x"]].to_csv(
    "laterality.csv", index=False)
log("wrote laterality.csv — done")
